In [4]:
import contextlib

import numpy as np
import numpy.typing as npt

In [5]:
def assign_symbols_from_range(
    vector: npt.NDArray | list[float],
    symbols: list[str] | list[float],
    weight: npt.NDArray | list[float] | list[int] | None = None,
) -> list[str] | list[float]:
    # Ensure vector is a numpy array
    vector = np.array(vector)

    # Check if all values in the vector are within [0, 1]
    if np.any(vector < 0) or np.any(vector > 1):
        msg = "All values in the vector must be within the range [0, 1]."
        raise ValueError(msg)

    # Determine the number of symbols
    num_symbols = len(symbols)

    # If a weight is provided, normalize the weight to sum up to 1
    if weight is not None:
        weight_arr = np.array(weight, dtype=np.float64)
        weight_arr /= weight_arr.sum()  # Normalize to get proportions
    else:
        # If no weight is provided, we use equal distribution
        weight_arr = np.ones(num_symbols) / num_symbols
        weight_arr = np.array(weight_arr, dtype=np.float64)

    # Create an array of cumulative weights
    cumulative_weight = np.cumsum(weight_arr)
    print(f"weight_arr={weight_arr} (sum={weight_arr.sum()})")
    print(f"cumulative_weight={cumulative_weight}")

    # Map the values to the corresponding symbols based on cumulative weight
    result = []
    for value in vector:
        # Find which bin the value falls into based on cumulative weight
        symbol_idx = np.searchsorted(cumulative_weight, value)
        result.append(symbols[symbol_idx])

    return result


# Example usage
vector = [0.1, 0.3, 0.5, 0.7, 0.9]
symbols = ["A", "B", "C", "D"]
weight = [1, 1, 1, 10]  # Optional weight argument

try:
    print(assign_symbols_from_range(vector, symbols, weight))
except ValueError as e:
    print(f"Error: {e}")

weight_arr=[0.07692308 0.07692308 0.07692308 0.76923077] (sum=1.0)
cumulative_weight=[0.07692308 0.15384615 0.23076923 1.        ]
['B', 'D', 'D', 'D', 'D']


In [ ]:
from collections.abc import Sequence
from typing import TypeVar

import numpy.typing as npt

T = TypeVar("T")


def assign_symbols_from_range(
    vector: npt.NDArray[np.float64] | Sequence[float],
    symbols: Sequence[T] | Sequence[Sequence[T]],
    weight: Sequence[float] | Sequence[Sequence[float]] | None = None,
) -> list[T]:
    vector = np.asarray(vector, dtype=np.float64)

    if np.any(vector < 0) or np.any(vector > 1):
        msg = "All values in the vector must be within [0, 1]."
        raise ValueError(msg)

    result: list[T] = []

    # Detect per-element symbols
    per_element = isinstance(symbols[0], (list, tuple, np.ndarray))

    for i, value in enumerate(vector):
        current_symbols: Sequence[T] = symbols[i] if per_element else symbols  # pyright: ignore[reportAssignmentType]

        num_symbols = len(current_symbols)

        # Handle weights
        if weight is not None:
            current_weight = (
                np.asarray(weight[i], dtype=np.float64)
                if per_element
                else np.asarray(weight, dtype=np.float64)
            )
            current_weight /= current_weight.sum()
        else:
            current_weight = (
                np.ones(num_symbols, dtype=np.float64) / num_symbols
            )

        cumulative_weight = np.cumsum(current_weight)

        symbol_idx = np.searchsorted(cumulative_weight, value)
        symbol_idx = min(symbol_idx, num_symbols - 1)

        result.append(current_symbols[symbol_idx])

    return result


# Example usage
vector = [0.1, 0.3, 0.5, 0.7, 0.9]
symbols = ["A", "B", "C", "D"]
weight = [1, 1, 1, 10]  # Optional weight argument

try:
    print(assign_symbols_from_range(vector, symbols, weight))
except ValueError as e:
    print(f"Error: {e}")

['B', 'D', 'D', 'D', 'D']


In [14]:
from ariel.body_phenotypes.robogen_lite.config import ALLOWED_ROTATIONS
from ariel.body_phenotypes.robogen_lite.config import ModuleType

print(ModuleType.keys)
print(ALLOWED_ROTATIONS[ModuleType["0"]])

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:4                                                                                    │
│                                                                                                  │
│   1 from ariel.body_phenotypes.robogen_lite.config import ALLOWED_ROTATIONS                      │
│   2 from ariel.body_phenotypes.robogen_lite.config import ModuleType                             │
│   3                                                                                              │
│ ❱ 4 print(ModuleType.keys)                                                                       │
│   5 print(ALLOWED_ROTATIONS[ModuleType["0"]])                                                    │
│   6                                                                                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
AttributeError: type object 'ModuleType' has no attribute 'keys'